In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf

In [ ]:
train_path = r"D:\excel files\Covid19-dataset\train"
test_path = r"D:\excel files\Covid19-dataset\test"

In [ ]:
categories = ['Covid', 'Normal', 'Viral Pneumonia']

img_size = 128

X = []
y = []

for category in categories:
    
    folder_path = os.path.join(train_path, category)
    label = categories.index(category)
    
    for img_name in os.listdir(folder_path):
        
        try:
            img_path = os.path.join(folder_path, img_name)
            
            img = cv2.imread(img_path)
            img = cv2.resize(img, (img_size, img_size))
            
            X.append(img)
            y.append(label)
            
        except:
            pass

In [ ]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
X = X / 255.0

LABLE ENCODER

In [ ]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

y = to_categorical(y)

print(y.shape)

TRAIN SPLIT

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

SAMPLE IMAGE

In [ ]:
plt.figure(figsize=(12,8))

for i in range(9):
    
    plt.subplot(3,3,i+1)
    
    plt.imshow(X[i])
    
    plt.title(categories[np.argmax(y[i])])
    
    plt.axis('off')

plt.show()

CLASS DISTRIBUTION PLOT

In [ ]:
class_count = [0,0,0]

for label in np.argmax(y, axis=1):
    class_count[label] += 1

sns.barplot(x=categories, y=class_count)

plt.title("Class Distribution")

plt.show()

MODEL 1 — BASIC CNN


In [ ]:
model1 = Sequential()

model1.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model1.add(MaxPooling2D(2,2))

model1.add(Conv2D(64, (3,3), activation='relu'))
model1.add(MaxPooling2D(2,2))

model1.add(Conv2D(128, (3,3), activation='relu'))
model1.add(MaxPooling2D(2,2))

model1.add(Flatten())

model1.add(Dense(128, activation='relu'))

model1.add(Dropout(0.5))

model1.add(Dense(3, activation='softmax'))

COMPILE MODEL

In [ ]:
model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

MODEL SUMMARY

In [ ]:
model1.summary()

EARLY STOPPING

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

TRAIN MODEL

In [ ]:
history1 = model1.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

ACCURACY GRAPH

In [ ]:
plt.plot(history1.history['accuracy'])
plt.plot(history1.history['val_accuracy'])

plt.title("Model Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(['Train','Validation'])

plt.show()

LOSS GRAPH

In [ ]:
plt.plot(history1.history['loss'])
plt.plot(history1.history['val_loss'])

plt.title("Model Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(['Train','Validation'])

plt.show()

EVALUATE MODEL

In [ ]:
loss, acc = model1.evaluate(X_val, y_val)

print("Accuracy :", acc)

PREDICTION

In [ ]:
y_pred = model1.predict(X_val)

y_pred = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_val, axis=1)

CONFUSION MATRIX

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories,
            yticklabels=categories)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

REPORT

In [ ]:
print(classification_report(y_true, y_pred))

MODEL 2 — TRANSFER LEARNING (VGG16)

In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

FREEZE LAYER

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

ADD CUSTOM LAYER

In [ ]:
x = Flatten()(base_model.output)

x = Dense(128, activation='relu')(x)

x = Dropout(0.5)(x)

output = Dense(3, activation='softmax')(x)

model2 = Model(inputs=base_model.input, outputs=output)

In [ ]:
model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history2 = model2.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

MODEL 3 — DATA AUGMENTATION + VGG16

IMAGE AUGMENTATION

In [ ]:
train_datagen = ImageDataGenerator(
    
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

FIT GENERATOR

In [ ]:
train_generator = train_datagen.flow(
    X_train,
    y_train,
    batch_size=32
)

TRAIN AUGMENTED MODEL

In [ ]:
history3 = model2.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=10
)

CLASS WEIGHTS

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(np.argmax(y_train, axis=1)),
    y=np.argmax(y_train, axis=1)
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

TRAIN WITH CLASS WEIGHT

In [ ]:
history = model1.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    class_weight=class_weights
)

MODEL SAVE

In [ ]:
model2.save("covid_xray_model.h5")

In [41]:
model2.save("covid19-xraymodel.keras")



In [ ]:
import keras
model2.export("covid_model")   



INFO:tensorflow:Assets written to: covid_model\assets


INFO:tensorflow:Assets written to: covid_model\assets


Saved artifact at 'covid_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_56')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2383403630416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2382370674576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403617168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403617552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403618896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403616784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403618128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403626384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403616208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403615248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2383403617936: TensorSpec(shape=(), 

In [ ]:
from keras.layers import TFSMLayer
model2 = TFSMLayer("covid_model", call_endpoint="serving_default")


#Streamlit app link

import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import time

st.set_page_config(
    page_title="COVID-19 Detection from Chest X-ray",
    page_icon="🩺",
    layout="centered"
)

st.markdown("""
<style>
.main {
    background-color: #f5f7fa;
}
.title {
    background-color: #0033cc;
    padding: 15px;
    border-radius: 10px;
    text-align: center;
    color: yellow;
    font-size: 40px;
    font-weight: bold;
}
.subtitle {
    text-align: center;
    font-size: 18px;
    color: gray;
    margin-bottom: 30px;
}
.result-box {
    padding: 20px;
    border-radius: 10px;
    text-align: center;
    font-size: 25px;
    font-weight: bold;
    margin-top: 20px;
}
</style>
""", unsafe_allow_html=True)

# LOAD MODEL
model = tf.keras.models.load_model("covid19-xraymodel.keras")
categories = ['Covid', 'Normal', 'Viral Pneumonia']

# HEADER
st.markdown('<div class="title">Covid-19 Detection App</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">A CNN based Covid Detection From Chest X-ray Classification System</div>', unsafe_allow_html=True)

# SIDEBAR
st.sidebar.image("https://cdn-icons-png.flaticon.com/512/2785/2785819.png", use_column_width=True)
st.sidebar.header("Upload X-ray Image")

uploaded_file = st.sidebar.file_uploader("Choose Chest X-ray Image", type=["jpg", "png", "jpeg"])

# MAIN SECTION
if uploaded_file is not None:
    try:
        # Always convert to RGB (fixes grayscale and RGBA issues)
        image = Image.open(uploaded_file).convert("RGB")

        st.image(image, caption="Uploaded Chest X-ray", use_column_width=True)

        # Preprocess
        img = np.array(image)
        img = cv2.resize(img, (128, 128))
        img = img / 255.0
        img = np.expand_dims(img, axis=0) 

        # Debug: show shape
        st.write(f"Image shape after preprocessing: {img.shape}")

        # Prediction
        with st.spinner("Analyzing X-ray Image..."):
            time.sleep(2)
            prediction = model.predict(img)

        predicted_class = categories[np.argmax(prediction)]
        confidence = np.max(prediction) * 100

        # Result display
        if predicted_class == "Covid":
            st.error(f"⚠️ Prediction : {predicted_class}\n\nConfidence : {confidence:.2f}%")
        elif predicted_class == "Normal":
            st.success(f"✅ Prediction : {predicted_class}\n\nConfidence : {confidence:.2f}%")
        else:
            st.warning(f"🫁 Prediction : {predicted_class}\n\nConfidence : {confidence:.2f}%")

        # Probability scores
        st.subheader("Prediction Probabilities")
        for i, category in enumerate(categories):
            st.write(f"{category} : {prediction[0][i]*100:.2f}%")
            st.progress(float(prediction[0][i]))

    except Exception as e:
        st.error(f"❌ Error processing image: {e}")
else:
    st.info("Upload a Chest X-ray Image from Sidebar")
